In [1]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import torch
import networkx as nx
import hvplot.pandas
import hvplot.xarray
import geoviews as gv
import holoviews as hv
import panel as pn

import xtensor as xt
import diffhydro as dh
import diffhydro.pipelines as dhp
from diffhydro.pipelines.base import init_inference_dl
from tqdm.auto import tqdm

from exp_helpers import (
    DEFAULT_DYNAMIC_KEYS, DEFAULT_STATIC_RUNOFF_KEYS, DEFAULT_ROUTING_STATIC_VAR,
    expand_dynamic_keys, expand_static_keys,
    define_splits, init_dataset,
)

In [2]:
EXP_NAME    = "default"
DEVICE      = "cuda:0"
DATA_DIR    = Path("../data")
RESULTS_DIR = Path("../results")

## Data loading

In [3]:
### Attention: changed to bassins.pkl instead of catchments.pkl as missing file

def load_local_graph(load_basins=False):
    """Load graph and keypoints from the local data/ folder."""
    g   = pd.read_pickle(DATA_DIR / "g.pkl")
    kp  = pd.read_pickle(DATA_DIR / "kp.pkl")
    df  = pd.DataFrame(dict(g.nodes)).T
    points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat))
    points["color"] = points["color"].fillna("black")
    catchments = gpd.GeoDataFrame(
        geometry=pd.read_pickle(DATA_DIR / "basins.pkl")).set_crs("epsg:4326")
    line = pd.read_pickle(DATA_DIR / "lines.pkl") if (DATA_DIR / "lines.pkl").exists() else None
    if load_basins:
        basins = gpd.GeoDataFrame(
            geometry=pd.read_pickle(DATA_DIR / "basins.pkl")).set_crs("epsg:4326") # originaly catchments.pkl
        return g, points, catchments, kp, basins
    return g, points, catchments, kp, line


def data_loading_local():
    """Load all model inputs from local data/ folder."""
    dynamic_var       = expand_dynamic_keys(DEFAULT_DYNAMIC_KEYS)
    runoff_static_var = expand_static_keys(DEFAULT_STATIC_RUNOFF_KEYS)
    routing_static_var = DEFAULT_ROUTING_STATIC_VAR

    g, _, _, kp, basins = load_local_graph(load_basins=True)

    routing_statics = pd.read_pickle(DATA_DIR / "routing_statics.pkl")[routing_static_var]
    routing_statics = (routing_statics - routing_statics.mean()) / routing_statics.std()
    routing_statics = routing_statics.fillna(0)

    runoff_statics = (
        xt.read_pickle(DATA_DIR / "runoff_statics.pkl", dims=["spatial", "variable"])
          .sel(variable=runoff_static_var)
    )
    runoff_statics = torch.nan_to_num(runoff_statics, 0)

    df_g = pd.DataFrame.from_dict(dict(g.nodes(data=True)), orient="index")
    channel_length = df_g["channel_length"] * 30 / 1000
    area           = df_g["catchment_area"]

    dyn_ds = xr.open_zarr(DATA_DIR / "dynamic_inp.zarr", consolidated=None)[dynamic_var].load()
    try:
        x = (
            xt.Dataset.from_xarray(dyn_ds)
              .to_datatensor(dim="variable")
              .expand_dims("batch")
              .to(dtype=torch.float)
              .sel(variable=dynamic_var)
              .transpose("batch", "spatial", "time", "variable")
        )
    finally:
        dyn_ds.close()

    y = (
        xt.open_datatensor(DATA_DIR / "discharges.zarr")
          .rename({"data_index": "spatial"})
          .expand_dims("batch")
          .to(dtype=torch.float)
          .transpose("batch", "spatial", "time")
    )
    y = y.assign_coords(spatial=y["spatial"].astype("int"))
    y = y.sel(time=x["time"])
    y = y.isel(spatial=~torch.isnan(y).all(dim=("time", "batch")))

    kp = kp.loc[kp["data_index"].isin(y["spatial"].to_pandas())]
    target_nodes = (
        kp.reset_index()
          .set_index("data_index")
          .loc[y["spatial"].to_pandas()]["grid_idxs"]
    )
    y = y.assign_coords(spatial=target_nodes)

    y_std  = y.std(dim=("time", "spatial"))
    x_mean = x.mean(dim=("time", "spatial"))
    x_std  = x.std(dim=("time", "spatial"))
    y = y / y_std
    x = (x - x_mean) / x_std
    x = torch.nan_to_num(x, nan=0.0)

    bad_kp = [504950665, 550839125, 552840355, 6041262397, 683814158, 677617648]
    kp_ = kp.loc[~kp.index.isin(bad_kp)]
    tr_nodes = kp_.loc[kp_.index.map(
        lambda n: not any(g.nodes[a]["is_dam"] for a in nx.ancestors(g, n))
    )].index
    all_nodes = kp.index

    splits = define_splits(g, tr_nodes, all_nodes, n_folds=10)

    return (g, x, y, x_mean, x_std, y_std,
            runoff_statics, routing_statics,
            channel_length, area, splits, kp, basins)

In [4]:
(g, x, y, x_mean, x_std, y_std,
 runoff_statics, routing_statics,
 channel_length, area, splits, kp, basins) = data_loading_local()

tr_nodes  = list(set().union(*[tr for tr, val, te in splits]))
all_nodes = list(set().union(*[te for tr, val, te in splits]))
print(f"Training nodes: {len(tr_nodes)},  gauged nodes: {len(all_nodes)}")

/lus/lfs1aip2/projects/u6t/vbrekke/xtensor/src/xtensor/datatensor.py:76: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return torch.as_tensor(values.to_numpy())


Training nodes: 318,  gauged nodes: 877


## Load model and run inference

In [9]:
inp_mlp_size  = len(routing_statics.columns)
inp_lstm_size = len(x["variable"]) + len(runoff_statics["variable"])

param_model = dhp.MLP(inp_mlp_size, 2)
model = dhp.RRModel(
    param_model,
    runoff_params={"hidden_size": 256, "num_layers": 2},
    input_size=inp_lstm_size,
    dt=1 / 24,
    max_delay=30,
    temp_res_h=24,
    irf_name="hayami",
    
).to(DEVICE)

model.load_state_dict(torch.load(RESULTS_DIR / f"{EXP_NAME}.pt", map_location=DEVICE))
model.eval()
print("Model loaded.")

Model loaded.


In [10]:
tr_ds = init_dataset(g, x, y, runoff_statics, routing_statics,
                     channel_length, area, tr_nodes,
                     init_window=365, pred_len=100,
                     irf_fn="hayami", include_index_diag=True)

module = dhp.RRModule(model, tr_ds, tr_ds, tr_ds,
                      batch_size=1, inference_batch_size=8, device=DEVICE)

y_tr, o_tr = module.extract_train(device=DEVICE, batch_size=1)
torch.cuda.empty_cache()

nse_tr = 1 - (((y_tr - o_tr) ** 2).mean("time") / y_tr.var("time"))
print(f"NSE train median: {nse_tr.median().item():.4f}")

y_obs  = y_tr.to_pandas().T       # DataFrame: time × node
y_pred = o_tr.to_pandas().T

  0%|          | 0/18 [00:00<?, ?it/s]

NSE train median: 0.9135


## Visualisation

In [11]:
class AnalysisPlot:
    def __init__(self, data, y, points, catchments, dams):
        self.g_points   = points
        self.y          = y
        self.data       = data
        self.catchments = catchments
        self.catchments["geometry"] = self.catchments.simplify(0.001)

        roots = [r for r, d in g.out_degree() if d == 0]
        self.root_basins = gv.Polygons(catchments.loc[roots]).opts(
            fill_alpha=0.0, line_color="black", line_width=3)
        self.dam_points = gv.Points(dams).opts(marker="square", color="black")
        self.input_widget = pn.widgets.Select(
            name="Results to show", options=list(data))

    def _convert_index(self, index):
        idx = index[0] if index else 0
        return self.g_points.index[idx]

    def _get_data(self, index):
        sim   = self.data[self.input_widget.value]
        idx   = self._convert_index(index)
        return pd.DataFrame({"sim": sim[idx], "gt": self.y[idx]})

    def plot_hydrograph(self, index):
        return self._get_data(index).hvplot()

    def plot_climato(self, index):
        idx = self._convert_index(index)
        x   = self._get_data(index)
        climato = x.groupby(x.index.month).mean()
        return climato.hvplot(title=str(idx), shared_axes=False).opts(
            ylim=(0, climato.max().max()))

    def polygon_plot(self, index):
        return gv.Polygons(
            [self.catchments.loc[self._convert_index(index), "geometry"]]
        ).opts(fill_alpha=0.1, line_color="black", line_width=3)

    def update_color(self, column="nse"):
        clim = (None, None) if column.endswith("prcp") else (0, 1)
        return gv.Points(self.g_points, vdims=[column]).opts(
            color=column, width=1000, height=1000, size=10,
            tools=["hover", "tap"], clim=clim, colorbar=True,
            cmap="coolwarm", nonselection_alpha=0.8, line_color="black")

    def plot(self):
        tile_sources  = {k: getattr(gv.tile_sources, k)
                         for k in dir(gv.tile_sources)
                         if isinstance(getattr(gv.tile_sources, k), gv.element.WMTS)}
        tile_selector = pn.widgets.Select(
            value="CartoLight", name="Tile Source", options=list(tile_sources))
        tiles      = hv.DynamicMap(pn.bind(lambda s: tile_sources[s], tile_selector))
        sel_widget = pn.widgets.Select(
            name="Model", options=self.g_points.columns.drop("geometry").tolist())
        point_plot = hv.DynamicMap(pn.bind(self.update_color, sel_widget))
        self.stream = hv.streams.Selection1D(source=point_plot)
        return pn.Column(
            pn.Row(
                pn.Column(
                    pn.Row(tile_selector, sel_widget),
                    tiles * self.root_basins * hv.DynamicMap(self.polygon_plot, streams=[self.stream])
                    * self.dam_points * point_plot,
                ),
                pn.Column(
                    pn.Card(self.input_widget, title="Select Model"),
                    pn.Card(hv.DynamicMap(self.plot_climato, streams=[self.stream]), title="Seasonality"),
                    pn.Card(hv.DynamicMap(self.plot_hydrograph, streams=[self.stream]), title="Time Series"),
                ),
            )
        )

In [14]:
nse_ser = nse_tr.to_pandas()

points = kp.loc[nse_ser.index, ["geometry"]].copy()
points["nse"] = nse_ser

dams = kp[kp.is_dam]
data = {"default": y_pred}

app = AnalysisPlot(data, y_obs, points, basins, dams)

In [15]:
app.plot()

BokehModel(combine_events=True, render_bundle={'docs_json': {'e52524f0-533b-4e1b-ab87-ec50333cc6ec': {'version…